# Importação e configuração do notebook

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
Catalogo_Name = "visagio"
DB_Bronze = "bronze"
DB_Silver = "silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {Catalogo_Name}.{DB_Silver}")
print(f"Banco de dados {Catalogo_Name}.{DB_Silver} criado/verificado com sucesso!")

Banco de dados visagio.silver criado/verificado com sucesso!


# Parte 1 - Tratamento de `tb_customers`

## Transformações Aplicadas

- **Tradução para português:** todas as colunas foram renomeadas para o padrão em português
- **Padronização em Upper Case:** campos `cidade`, `estado` e `nome_consumidor` convertidos para maiúsculas, facilitando manipulação e evitando inconsistências de case
- **Deduplicação por ID:** utilizada **Window Function** particionada por `id_consumidor` e ordenada de forma decrescente por `data_ingestao`, garantindo que apenas o registro mais recente seja mantido

In [0]:
def create_dim_consumidores(df):
    try:
        window_f = Window.partitionBy("customer_id") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_f))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("customer_id").alias("id_consumidor"),
                F.col("customer_unique_id").alias("id_consumidor_unico"),
                F.col("customer_zip_code_prefix").alias("prefixo_cep"),
                F.upper(F.col("customer_name")).alias("nome_consumidor"),
                F.upper(F.col("customer_city")).alias("cidade"),
                F.upper(F.col("customer_state")).alias("estado"),
                F.col("customer_gender").alias("genero"),
                F.col("customer_birth_date").alias("data_nascimento"),
                F.col("customer_age").alias("idade")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_consumidores"
        # Salva a tabela no formato Delta Lake
        # Note o uso de repartition para eficiencia futura dos dados, dessa forma evitando tamanhos de arquivos diferentes
        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")
    except Exception as e:
        print(f"Erro ao criar silver.dim_consumidores: {str(e)}")

create_dim_consumidores(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_customers"))

Tabela visagio.silver.dim_consumidores atualizada com sucesso!


# Parte 2 - Tratamento de `tb_orders`

## Transformações Aplicadas

- **Tradução de Status:** mapeamento completo dos status de pedidos do inglês para português
- **Cálculo de Métricas de Entrega:**
  - `tempo_entrega_dias`: diferença entre entrega ao cliente e data da compra
  - `tempo_entrega_estimado_dias`: diferença entre estimativa e data da compra
  - `diferenca_entrega_dias`: atraso ou antecipação da entrega
- **Indicador de Prazo:** campo `entrega_no_prazo` categorizando entregas (Sim/Não/Não Entregue)
- **Deduplicação por ID:** Window Function garantindo registro mais recente por `order_id`
- **Particionamento:** uso de `repartition(4)` para balancear distribuição dos arquivos Delta

In [0]:
def create_fat_pedidos(df):
    try:
        status_map = {
            "delivered"   : "entregue",
            "canceled"    : "cancelado",
            "shipped"     : "enviado",
            "processing"  : "processando",
            "invoiced"    : "faturado",
            "unavailable" : "indisponível",
            "created"     : "criado",
            "approved"    : "aprovado"
        }
        mapping_expr = F.create_map(
            *[F.lit(x) for pair in status_map.items() for x in pair]
        )

        df_silver = (
            df
            .withColumn("status_pt", mapping_expr[F.col("order_status")])
            .withColumn(
                "tempo_entrega_dias",
                F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
            )
            .withColumn(
                "tempo_entrega_estimado_dias",
                F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp"))
            )
            .withColumn(
                "diferenca_entrega_dias",
                F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
            )
            .withColumn(
                "entrega_no_prazo",
                F.when(F.col("order_status") != "delivered", "Não Entregue")
                 .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
                 .otherwise("Não")
            )
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("customer_id").alias("id_consumidor"),
                F.col("status_pt").alias("status"),
                F.col("order_purchase_timestamp").alias("data_compra"),
                F.col("order_approved_at").alias("data_aprovacao"),
                F.col("order_delivered_carrier_date").alias("data_entrega_transportadora"),
                F.col("order_delivered_customer_date").alias("data_entrega_cliente"),
                F.col("order_estimated_delivery_date").alias("data_estimada_entrega"),
                F.col("tempo_entrega_dias"),
                F.col("tempo_entrega_estimado_dias"),
                F.col("diferenca_entrega_dias"),
                F.col("entrega_no_prazo")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pedidos"

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao criar silver.fat_pedidos: {str(e)}")

create_fat_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_orders"))

✅ Tabela visagio.silver.fat_pedidos atualizada com sucesso!


# Parte 3 - Tratamento de `tb_order_items`

## Transformações Aplicadas

- **Tradução para português:** renomeação de colunas para padrão PT-BR
- **Deduplicação:** Window Function por `order_id` e `order_item_id` mantendo registro mais recente
- **Granularidade:** preservação do nível item dentro do pedido (relação 1:N)
- **Preços separados:** mantem `preco_BRL` e `preco_frete` como colunas independentes
- **Particionamento:** `repartition(4)` para distribuição equilibrada

In [0]:
def create_fat_itens_pedidos(df):
    try:
        window_spec = Window.partitionBy("order_id", "order_item_id") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("order_item_id").alias("id_item"),
                F.col("product_id").alias("id_produto"),
                F.col("seller_id").alias("id_vendedor"),
                F.col("shipping_limit_date").alias("data_limite_envio"), 
                F.col("price").alias("preco_BRL"),
                F.col("freight_value").alias("preco_frete")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_itens_pedidos"

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"Erro ao criar silver.fat_itens_pedidos: {str(e)}")

create_fat_itens_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_items"))

Tabela visagio.silver.fat_itens_pedidos atualizada com sucesso!


# Parte 4 - Tratamento de `tb_order_payments`

## Transformações Aplicadas

- **Tradução de Métodos de Pagamento:** mapeamento para português (Cartão de Crédito, Boleto, Voucher, etc.)
- **Deduplicação:** Window Function por `order_id` e `payment_sequential` garantindo consistência
- **Granularidade de Parcelas:** preservação da sequência de pagamento para casos de múltiplos métodos
- **Valores monetários:** manutenção de precisão dos valores de pagamento
- **Particionamento:** `repartition(4)` para otimização de I/O

In [0]:
def create_fat_pagamentos_pedidos(df):
    try:
        window_spec = Window.partitionBy("order_id", "payment_sequential") \
                            .orderBy(F.col("timestamp_ingestion").desc())

        payment_map = {
            "credit_card" : "Cartão de Crédito",
            "boleto"      : "Boleto",
            "voucher"     : "Voucher",
            "debit_card"  : "Cartão de Débito",
            "not_defined" : "Não Definido"
        }

        mapping_expr = F.create_map(
            *[F.lit(x) for pair in payment_map.items() for x in pair]
        )

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .withColumn("tipo_pagamento", mapping_expr[F.col("payment_type")])
            .select(
                F.col("order_id").alias("id_pedido"),
                F.col("payment_sequential").alias("sequencial_pagamento"),
                F.col("tipo_pagamento"),
                F.col("payment_installments").alias("parcelas"),
                F.col("payment_value").alias("valor_pagamento")
            )
        )

        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pagamentos_pedidos"

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"Tabela {target_table} atualizada com sucesso!")

    except Exception as e:
        print(f"Erro ao criar silver.fat_pagamentos_pedidos: {str(e)}")

create_fat_pagamentos_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_payments"))

Tabela visagio.silver.fat_pagamentos_pedidos atualizada com sucesso!


# Parte 5 - Tratamento de `tb_order_reviews`

## Transformações Aplicadas

- **Conversão de Timestamps:** uso de `try_to_timestamp` para segurança na conversão
- **Filtros de Qualidade:**
  - Remove registros sem `order_id` (inválidos)
  - Valida timestamps futuros
- **Tratamento de Nulos:** campos vazios preenchidos com "Sem título" e "Sem comentário"
- **Tradução:** renomeação para padrão português
- **Particionamento:** `repartition(4)` para balanceamento de arquivos Delta

In [0]:
def create_fat_avaliacoes_pedidos(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_avaliacoes_pedidos"

        df_silver = (
            df
            .withColumn("review_creation_date",    F.try_to_timestamp("review_creation_date"))
            .withColumn("review_answer_timestamp", F.try_to_timestamp("review_answer_timestamp"))

            .filter(F.col("order_id").isNotNull())

            .filter(
                F.col("review_creation_date").isNull() | 
                (F.col("review_creation_date") <= F.current_timestamp())
            )

            .withColumn("review_comment_title",
                F.when(
                    F.col("review_comment_title").isNull() | (F.trim(F.col("review_comment_title")) == ""),
                    "Sem título"
                ).otherwise(F.col("review_comment_title"))
            )
            .withColumn("review_comment_message",
                F.when(
                    F.col("review_comment_message").isNull() | (F.trim(F.col("review_comment_message")) == ""),
                    "Sem comentário"
                ).otherwise(F.col("review_comment_message"))
            )

            .select(
                F.col("review_id").alias("id_avaliacao"),
                F.col("order_id").alias("id_pedido"),
                F.col("review_score").alias("nota_avaliacao"),
                F.col("review_comment_title").alias("titulo_avaliacao_comentario"),
                F.col("review_comment_message").alias("mensagem_avaliacao_comentario"),
                F.col("review_creation_date").alias("data_criacao_avaliacao"),
                F.col("review_answer_timestamp").alias("data_resposta_avaliacao")
            )
        )

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f" Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")
    except Exception as e:
        print(f" Erro ao criar silver.fat_avaliacoes_pedidos: {str(e)}")

create_fat_avaliacoes_pedidos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_order_reviews"))

 Tabela visagio.silver.fat_avaliacoes_pedidos atualizada com sucesso! (99224 registros)



# Parte 6 - Tratamento de `tb_products`

## Transformações Aplicadas

- **Dimensão de Produto:** tabela dimensional contendo atributos descritivos dos produtos
- **Deduplicação:** Window Function por `product_id` mantendo versão mais recente
- **Atributos Físicos:** preservação de peso, dimensões (comprimento, altura, largura)
- **Metadados:** quantidade de fotos, tamanho de nome e descrição
- **Tradução:** padronização de nomes de colunas em português
- **Particionamento:** `repartition(4)` para otimização de consultas futuras

In [0]:
def create_dim_produtos(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_produtos"
        window_spec = Window.partitionBy("product_id").orderBy(
            F.col("data_ingestao").desc(),
        )

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("product_id").alias("id_produto"),
                F.col("product_name").alias("nome_produto"),
                F.col("product_category_name").alias("categoria_produto"),
                F.col("product_name_lenght").alias("tamanho_nome_produto"),
                F.col("product_description_lenght").alias("tamanho_descricao_produto"),
                F.col("product_photos_qty").alias("quantidade_fotos"),
                F.col("product_weight_g").alias("peso_produto_gramas"),
                F.col("product_length_cm").alias("comprimento_centimetros"),
                F.col("product_height_cm").alias("altura_centimetros"),
                F.col("product_width_cm").alias("largura_centimetros")
            )
        )

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_produtos: {str(e)}")

create_dim_produtos(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_products"))

❌ Erro ao criar silver.dim_produtos: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `data_ingestao` cannot be resolved. Did you mean one of the following? [`timestamp_ingestion`, `product_id`, `product_name`, `product_length_cm`, `product_weight_g`]. SQLSTATE: 42703;
'Filter '`==`('rn, 1)
+- 'Project [product_id#57141, product_name#57142, product_category_name#57143, product_name_lenght#57144, product_description_lenght#57145, product_photos_qty#57146, product_weight_g#57147, product_length_cm#57148, product_height_cm#57149, product_width_cm#57150, timestamp_ingestion#57151, row_number() windowspecdefinition(product_id#57141, 'data_ingestao DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rn#57152]
   +- SubqueryAlias visagio.bronze.tb_products
      +- Relation visagio.bronze.tb_products[product_id#57141,product_name#57142,product_category_name#57143,product_name_lenght#57144,product_description_lenght#5

# Parte 7 - Tratamento de `tb_sellers`

## Transformações Aplicadas

- **Dimensão de Vendedor:** tabela dimensional com informações dos vendedores
- **Padronização Geográfica:** campos `cidade` e `estado` convertidos para maiúsculas
- **Deduplicação:** Window Function por `seller_id` garantindo registro único
- **Localização:** preservação de prefixo CEP para análises regionais
- **Tradução:** padrão de nomenclatura em português
- **Particionamento:** `repartition(4)` para distribuição eficiente

In [0]:
def create_dim_vendedores(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_vendedores"

        window_spec = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

        df_silver = (
            df
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .select(
                F.col("seller_id").alias("id_vendedor"),
                F.col("seller_zip_code_prefix").alias("prefixo_cep"),
                F.col("seller_name").alias("nome_vendedor"),
                F.upper(F.col("seller_city")).alias("cidade"),
                F.upper(F.col("seller_state")).alias("estado")
            )
        )

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_vendedores: {str(e)}")

create_dim_vendedores(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_sellers"))

✅ Tabela visagio.silver.dim_vendedores atualizada com sucesso! (3095 registros)



# Parte 8 - Tratamento de `tb_product_category_name_translation`

## Transformações Aplicadas

- **Tabela de Tradução:** mapeamento bilingüe português-inglês das categorias
- **Uso:** permite joins com `dim_produtos` para obter nomes em ambos os idiomas
- **Simplicidade:** sem deduplicação necessária (tabela de referência estática)
- **Tradução:** nomenclatura padronizada PT/EN
- **Particionamento:** `repartition(4)` mesmo sendo pequena, mantendo consistência do padrão

In [0]:
def create_dim_categoria_produtos_traducao(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_categoria_produtos_traducao"

        df_silver = (
            df
            .select(
                F.col("product_category_name").alias("nome_produto_pt"),
                F.col("product_category_name_english").alias("nome_produto_en")
            )
        )

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_categoria_produtos_traducao: {str(e)}")

create_dim_categoria_produtos_traducao(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_product_category_name_translation"))

✅ Tabela visagio.silver.dim_categoria_produtos_traducao atualizada com sucesso! (71 registros)



# Parte 9 - Tratamento de `tb_cotacao_dolar`

## Transformações Aplicadas

- **Calendário Contínuo:** geração de todas as datas no range usando `sequence()`
- **Cotação de Fechamento:** Window Function para pegar última cotação do dia
- **Forward Fill:** preenchimento de finais de semana com cotação da sexta anterior usando `last(..., ignorenulls=True)`
- **Uso:** permite joins diretos por data sem perder registros em dias não úteis
- **Critério:** `rowsBetween(Window.unboundedPreceding, 0)` para propagação sequencial
- **Particionamento:** `repartition(4)` para consistência do padrão

In [0]:
def create_dim_cotacao_dolar(df):
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.dim_cotacao_dolar"
        datas = df.agg(
            F.min(F.to_date("data_hora_cotacao")).alias("data_min"),
            F.max(F.to_date("data_hora_cotacao")).alias("data_max")
        ).collect()[0]

        data_min = datas["data_min"]
        data_max = datas["data_max"]

        print(f"Range: {data_min} → {data_max}")

        # Calendário contínuo — data_date como DateType para o join
        df_calendario = spark.sql(f"""
            SELECT explode(sequence(
                to_date('{data_min}'),
                to_date('{data_max}'),
                interval 1 day
            )) as data_date
        """)

        # Cotação de fechamento do dia
        window_fechamento = Window.partitionBy(F.to_date("data_hora_cotacao")).orderBy(F.col("data_hora_cotacao").desc())

        df_cotacao_fechamento = (
            df
            .withColumn("rn", F.row_number().over(window_fechamento))
            .filter(F.col("rn") == 1)
            .drop("rn")
            .withColumn("data_date", F.to_date("data_hora_cotacao"))
        )

        df_completo = df_calendario.join(df_cotacao_fechamento, on="data_date", how="left")

        # Forward fill
        window_fill = Window.orderBy("data_date").rowsBetween(Window.unboundedPreceding, 0)

        df_silver = (
            df_completo
            .withColumn("cotacao_compra",    F.last("cotacao_compra",    ignorenulls=True).over(window_fill))
            .withColumn("data_hora_cotacao", F.last("data_hora_cotacao", ignorenulls=True).over(window_fill))
            .select(
                F.date_format(F.col("data_date"), "dd-MM-yyyy").alias("data_cotacao"),
                F.date_format(F.col("data_hora_cotacao"), "dd-MM-yyyy").alias("data_hora_cotacao"),
                F.col("cotacao_compra").alias("cotacao_compra")
            )
        )

        df_silver.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_silver.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.dim_cotacao_dolar: {str(e)}")
create_dim_cotacao_dolar(spark.table(f"{Catalogo_Name}.{DB_Bronze}.tb_cotacao_dolar"))

Range: 2016-09-05 → 2018-12-31


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Tabela visagio.silver.dim_cotacao_dolar atualizada com sucesso! (848 registros)



# Parte 10 - Tratamento de `fat_pedido_total`

## Transformações Aplicadas

- **Fato Agregado:** consolidação de múltiplas tabelas (pedidos + pagamentos + cotação)
- **Agregação de Pagamentos:** soma de todos os valores pagos por pedido
- **Conversão Cambial:** cálculo automático BRL → USD usando cotação da data do pedido
- **Join Temporal:** relacionamento pela data da compra com `dim_cotacao_dolar`
- **Moedas Duais:** `valor_total_pago_brl` e `valor_total_pago_usd` para análises multidimensionais
- **Particionamento:** `repartition(4)` para otimização de queries analíticas

In [0]:
def create_fat_pedido_total():
    try:
        target_table = f"{Catalogo_Name}.{DB_Silver}.fat_pedido_total"

        df_pedidos    = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_pedidos")
        df_pagamentos = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_pagamentos_pedidos")
        df_cotacao    = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_cotacao_dolar")

        # Agrega pagamentos por pedido
        df_pagamentos_agg = (
            df_pagamentos
            .groupBy("id_pedido")
            .agg(F.round(F.sum("valor_pagamento"), 2).alias("valor_total_pago_brl"))
        )

        # Join pedidos com pagamentos
        df_joined = (
            df_pedidos
            .join(df_pagamentos_agg, on="id_pedido", how="left")
        )

        # Join com cotação do dólar pela data do pedido
        df_final = (
            df_joined
            .join(
                df_cotacao,
                on=F.to_date(df_joined["data_compra"]) == F.to_date(df_cotacao["data_cotacao"], "dd-MM-yyyy"),
                how="left"
            )
            .drop(df_cotacao["data_cotacao"])    # remove coluna auxiliar do join
            .withColumn("valor_total_pago_usd",
                F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2)
            )
            .select(
                F.col("id_pedido"),
                F.col("id_consumidor"),
                F.col("status"),
                F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
                F.round(F.col("valor_total_pago_usd"), 2).alias("valor_total_pago_usd"),
                F.col("data_compra").alias("data_pedido")
            )
        )

        df_final.repartition(4).write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_final.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar silver.fat_pedido_total: {str(e)}")

create_fat_pedido_total()

✅ Tabela visagio.silver.fat_pedido_total atualizada com sucesso! (99441 registros)



# Parte 11 - Otimização de Tabelas Delta

## Operações Aplicadas

- **OPTIMIZE:** compactação de arquivos pequenos em arquivos maiores para melhor desempenho
- **ZORDER:** reorganização física dos dados por `id_pedido` (e `data_pedido` no fat_pedido_total)
- **Objetivo:** acelerar queries que filtram por essas colunas, co-localizando dados relacionados
- **Tabelas Otimizadas:** todas as tabelas fato que usam `id_pedido` como chave de consulta
- **Benefícios:** redução de data skipping e menor número de arquivos lidos por query

In [0]:
tabelas_otimizar = [
    "fat_pedidos",
    "fat_itens_pedidos",
    "fat_pagamentos_pedidos",
    "fat_pedido_total",
]

for tabela in tabelas_otimizar:
    try:
        # id_pedido existe em todas, data_pedido só no fat_pedido_total
        zorder_cols = "id_pedido, data_pedido" if tabela == "fat_pedido_total" else "id_pedido"

        spark.sql(f"""
            OPTIMIZE {Catalogo_Name}.{DB_Silver}.{tabela}
            ZORDER BY ({zorder_cols})
        """)
        print(f"✅ OPTIMIZE concluído: {tabela}")

    except Exception as e:
        print(f"❌ Erro ao otimizar {tabela}: {str(e)}")

✅ OPTIMIZE concluído: fat_pedidos
✅ OPTIMIZE concluído: fat_itens_pedidos
✅ OPTIMIZE concluído: fat_pagamentos_pedidos
✅ OPTIMIZE concluído: fat_pedido_total
